# LongHealth: minimal public-data loader

LongHealth contains 20 fictional longitudinal clinical cases and 400 multiple-choice questions. This notebook downloads the official benchmark and inspects its structure without reproducing narratives or question contents.

The authors request that users avoid publicly sharing examples to reduce contamination of future model-training data. Accordingly, all displays below are aggregate, schema-only, or redacted.

Paper: https://arxiv.org/abs/2401.14490  
Dataset: https://github.com/kbressem/LongHealth

The benchmark is research data and must not be used for clinical decision-making.

In [ ]:
import json
from collections import Counter
from urllib.request import Request, urlopen

DATA_URL = (
    "https://raw.githubusercontent.com/kbressem/LongHealth/"
    "main/data/benchmark_v5.json"
)
request = Request(DATA_URL, headers={"User-Agent": "longhealth-colab-demo"})
with urlopen(request, timeout=60) as response:
    benchmark = json.load(response)

print(f"Loaded {len(benchmark)} fictional patient cases")
print(DATA_URL)

In [ ]:
# Validate the documented core schema without displaying benchmark contents.
required_case_fields = {"texts", "name", "birthday", "diagnosis", "questions", "canaray-string"}
required_question_fields = {
    "No", "question", "answer_a", "answer_b", "answer_c",
    "answer_d", "answer_e", "correct", "answer_location",
}

for patient_id, case in benchmark.items():
    missing_case = required_case_fields - case.keys()
    assert not missing_case, f"{patient_id} missing case fields: {sorted(missing_case)}"
    assert isinstance(case["texts"], dict)
    assert isinstance(case["questions"], list)
    for question in case["questions"]:
        missing_question = required_question_fields - question.keys()
        assert not missing_question, f"{patient_id} missing question fields: {sorted(missing_question)}"

print("Schema validated for every case and question.")
print("Case fields:", sorted(required_case_fields))
print("Question fields:", sorted(required_question_fields))

In [ ]:
# Aggregate statistics only—no clinical narrative or QA content is printed.
total_documents = sum(len(case["texts"]) for case in benchmark.values())
total_questions = sum(len(case["questions"]) for case in benchmark.values())
questions_per_case = Counter(len(case["questions"]) for case in benchmark.values())
documents_per_case = Counter(len(case["texts"]) for case in benchmark.values())

print("Cases:", len(benchmark))
print("Clinical documents:", total_documents)
print("Questions:", total_questions)
print("Questions per case distribution:", dict(sorted(questions_per_case.items())))
print("Documents per case distribution:", dict(sorted(documents_per_case.items())))
assert len(benchmark) == 20
assert total_questions == 400

In [ ]:
# Two safe structural examples. Names, dates, diagnoses, text, questions, options, and labels stay hidden.
def safe_summary(patient_id, case):
    return {
        "patient_id": patient_id,
        "number_of_documents": len(case["texts"]),
        "document_character_counts": {key: len(value) for key, value in case["texts"].items()},
        "number_of_questions": len(case["questions"]),
        "question_numbers": [question["No"] for question in case["questions"]],
    }

for patient_id in list(benchmark)[:2]:
    print(json.dumps(safe_summary(patient_id, benchmark[patient_id]), indent=2))

In [ ]:
# Verify the actual annotation format: parallel lists of normalized start/end positions.
checked_spans = 0
for patient_id, case in benchmark.items():
    for question in case["questions"]:
        for text_id, span in question["answer_location"].items():
            assert text_id in case["texts"], (patient_id, text_id)
            starts, ends = span["start"], span["end"]
            assert isinstance(starts, list) and isinstance(ends, list)
            assert len(starts) == len(ends)
            for start, end in zip(starts, ends):
                assert isinstance(start, (int, float)) and isinstance(end, (int, float))
                assert 0.0 <= start <= end <= 1.0
                checked_spans += 1

print(f"Validated {checked_spans} normalized answer-location spans without displaying their contents.")

## Responsible inspection

For local research, you can inspect `benchmark[patient_id]['texts']` and `benchmark[patient_id]['questions']`. Avoid placing their contents in public notebooks, reports, prompts that may be logged for training, or redistributed files. Cite the original paper and repository when publishing results.